Okay we start with some fundementals thats must be know 
First of all, what is risk-neutral pricing?
Under the no-arbitrage assumption, risk-neutral pricing consists of replicating the derivative’s payoff using a self-financing portfolio made of the underlying asset and a risk-free asset.  
Because the replicating portfolio and the derivative produce the same payoff, they must have the same value today. Otherwise, we could make an arbitrage profit by going long the cheaper one and short the more expensive one.  
Therefore, the derivative price does not depend on the real-world probability of the underlying asset going up or down. Instead, it can be written as the discounted expected payoff under a special probability measure called the risk-neutral measure.

Both \(p\) and \(q\) are probabilities of the stock going up. The difference is that \(p\) is the probability under the real-world measure \(P\), while \(q\) is the probability of the same event under the risk-neutral measure \(Q\). They generally do not have the same numerical value.

CRR parametrazation.
We will split the interval [0,T] into N steps . so t_k+1-t_k=dt=T/N.
sigma: volatility.

T: time to expiration.
r: risk free rate.
K: strike.
S: stock price.

u=exp(sqrt(dt)*sigma) up coef.

d=exp(-sqrt(dt)*sigma) down coef.

p=exp(rdt)-d/u-d rsik netural probability.

combinatory tree : S(i,j)=S0*u^i*d^i-j.


the price of the option S0=exp(-rT)*E_Q[payyoff(S_T)].
V=exp(-rdt)*(p*V_up+(1-p)*V_down).

for the american option we compare V against the immidiate payyoff if its greater we exercice.

As ``N -> infinity`` the binomial European price converges to Black-Scholes.


In [1]:
from math import sqrt , exp
import numpy as np

In [60]:
#lets craft the binomial Tree

def BTree(sigma:float,r:float,T:float,N:int,S0:float):
    dt=T/N
    u=exp(sigma*sqrt(dt)) #up coef
    d=exp(-sigma*sqrt(dt)) #down coef
    p=(exp(r*dt)-d)/(u-d) #risk netural probability
    S=np.zeros((N+1,N+1))
    S[0,0]=S0
    z=1
    for j in range (1,N+1):
        for i in range (z):
            #when its up and when its down
            S[i,j]=S[i,j-1]*u
            S[i+1,j]=S[i,j-1]*d
        z+=1            
    
    return S,p
    
def Payoff(K:float,Spot:np.array,OptionType:chr):
    if OptionType=="call":
        return np.maximum(Spot-K,0)
    return np.maximum(K-Spot,0)
    
    
def PriceEurope(sigma:float,r:float,T:float,N:int,S0:float,K:float,OptionType:chr):
    Spots,p=BTree(sigma,r,T,N,S0)
    dt=T/N
    V=np.zeros((N+1,N+1)) #V[i,N] we need to fill this last column

        #first we fill in the N+1 column with the payoffs
    for i in range (N+1):
        V[i,N]=Payoff(K,Spots[i,N],OptionType)
            
    for j in range (N-1,-1,-1): 
        for i in range (j+1): #backward induction each time we remove a row and calculate V
            V[i,j]=exp(-r*dt)*(p*V[i,j+1]+(1-p)*V[i+1,j+1])
    return V

def PriceAmerican(sigma:float,r:float,T:float,N:int,S0:float,K:float,OptionType:chr):
    spots,p=BTree(sigma,r,T,N,S0)
    dt=T/N
    V=np.zeros((N+1,N+1))
    for i in range(N+1):
        V[i,N]=Payoff(K,spots[i,N],OptionType)
    for j in range(N-1,-1,-1):
        for i in range(j+1):
            V[i,j]=np.maximum(Payoff(K,spots[i,j],OptionType),exp(-r*dt)*(p*V[i,j+1]+(1-p)*V[i+1,j+1]))
    return V



In [75]:
#testing:
S0=36
T=1.0
sigma=0.2
r=0.06
N=1000
K=35

In [76]:
print(PriceEurope(sigma,r,T,N,S0,K,"call")[0,0])

4.527498368435431


In [63]:
print(PriceAmerican(sigma,r,T,N,S0,K,"put")[0,0])

1.6578184137816894


Under the Q mesure the stock follows this EDP: \
dS_t=r*S_t*dt+sigma*S_t*dW_t where W is a brownian motion.

S_t=S_0*exp((r-sigma^2\2)t+sigma*sqrt{T}*Z),where Z~N(0,1). 

and the payyoff f(S_t)=exp(-r*(T-t)*E_Q[ f_T ]).\
so for the europian call price we have:\n

C_t=exp(-r*(T-t))*(exp(r(T-t))*S*N(d1)-K*N(d2)) where :\

d1=(ln(S/k)+(r+sigma^2/2)(T-t))/sigma*sqrt(T-t)\
d2=(ln(S/k)+(r-sigma^2/2)(T-t))/sigma*sqrt(T-t)\
N(x)=1/sqrt(2*pi) * integral:-infini to x (exp(-u^^2)du)\


In [79]:
import scipy.integrate as integrate

def N(u):
        return (integrate.quad(lambda x :(1/np.sqrt(2*np.pi))*exp((-x**2)/2),-np.inf,u))[0]
    
def BlackScholes(sigma,r,T,K,S,OptionType):
    d1=(np.log(S/K)+(r+(sigma**2)/2)*T)/(sigma*sqrt(T))
    d2=(np.log(S/K)+(r-(sigma**2)/2)*T)/(sigma*sqrt(T))
    
    if OptionType=="call":
        V=exp(-r*T)*(exp(r*T)*S*N(d1)-K*N(d2))
    if OptionType=="put":
        V=exp(-r*T)*(K*N(-d2)-exp(r*T)*S*N(-d1))
    return V
    

In [80]:
print(BlackScholes(sigma,r,T,K,S0,"call"))

4.527296534725704
